In [ ]:
import json
import os
from collections import defaultdict, OrderedDict
import pandas as pd
from IPython.display import display, HTML

# ========= CONFIG =========
# Add all your JSONs here (absolute or relative paths)
JSON_FILES = [
    "predictions/gold_only_pred.json",
    "predictions/silver_only_pred.json",
    "predictions/silver_to_gold_pred.json"
]

ENTITY_GROUPS = OrderedDict([
    ("Recipient", {1, 2}),
    ("Authority", {3, 4}),
    ("Legal Object", {5, 6}),
    ("Legal Action", {7, 8}),
    ("Legal Basis", {9, 10}),
])

# === AUTHORITY NAME MAPPING ===
AUTHORITY_MAP = {
    "kansspelau": "KSA",
    "open.overh": "RIJ",
    "puc.overhe": "ANVS",
    "www.acm.nl": "ACM",
    "www.dnb.nl": "DNB",
    "www.evenem": "ROT",
    "www.provin": "DRE",
}

def map_authority(auth_id):
    """Map the first 10 characters of id to readable short name."""
    return AUTHORITY_MAP.get(auth_id, auth_id)  # default to original if not found

def safe_div(n, d): 
    return n / d if d else 0.0

def compute_metrics(counts_dict):
    """Compute precision, recall, F1 given TP, FP, FN counts."""
    metrics = {}
    for k, v in counts_dict.items():
        TP, FP, FN = v["TP"], v["FP"], v["FN"]
        P = safe_div(TP, TP + FP)
        R = safe_div(TP, TP + FN)
        F1 = safe_div(2 * P * R, P + R)
        metrics[k] = (P, R, F1)
    return metrics

def process_json(path):
    """Compute all metrics for one JSON file."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    entity_counts = {ent: {"TP": 0, "FP": 0, "FN": 0} for ent in ENTITY_GROUPS}
    auth_ent_counts = defaultdict(lambda: {ent: {"TP": 0, "FP": 0, "FN": 0} for ent in ENTITY_GROUPS})

    for item in data:
        gold = item["ner_tags"]
        pred = item["pred_tags"]
        auth = map_authority(item["id"][:10])
        for ent, tagset in ENTITY_GROUPS.items():
            for g, p in zip(gold, pred):
                g_in, p_in = g in tagset, p in tagset
                if g_in and p_in:
                    entity_counts[ent]["TP"] += 1
                    auth_ent_counts[auth][ent]["TP"] += 1
                elif (not g_in) and p_in:
                    entity_counts[ent]["FP"] += 1
                    auth_ent_counts[auth][ent]["FP"] += 1
                elif g_in and (not p_in):
                    entity_counts[ent]["FN"] += 1
                    auth_ent_counts[auth][ent]["FN"] += 1

    entity_metrics = compute_metrics(entity_counts)
    auth_ent_metrics = {}
    authority_macro = {}
    for auth, counts in auth_ent_counts.items():
        per_entity = compute_metrics(counts)
        auth_ent_metrics[auth] = per_entity
        authority_macro[auth] = sum(m[2] for m in per_entity.values()) / len(per_entity)

    # Build one table for this file
    rows = []
    for auth, ents in auth_ent_metrics.items():
        row = {"Authority": auth}
        for ent, (P, R, F1) in ents.items():
            row[f"{ent} P"] = round(P, 3)
            row[f"{ent} R"] = round(R, 3)
            row[f"{ent} F1"] = round(F1, 3)
        row["Macro-F1"] = round(authority_macro[auth], 3)
        rows.append(row)

    # Make DataFrame
    columns = ["Authority"]
    for ent in ENTITY_GROUPS.keys():
        columns += [f"{ent} P", f"{ent} R", f"{ent} F1"]
    columns += ["Macro-F1"]

    df = pd.DataFrame(rows)[columns].sort_values("Macro-F1", ascending=False).reset_index(drop=True)
    return df

# ========= MAIN LOOP =========
combined_results = []

for path in JSON_FILES:
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        continue

    df = process_json(path)
    model_name = os.path.splitext(os.path.basename(path))[0]
    df["Model"] = model_name  # track source

    # Display nicely
    display(HTML(f"<h3>Results for: {model_name}</h3>"))
    display(df.style.format(precision=3).set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'center')]},
         {'selector': 'td', 'props': [('text-align', 'center')]}]
    ))
    combined_results.append(df)

,Authority,Recipient P,Recipient R,Recipient F1,Authority P,Authority R,Authority F1,Legal Object P,Legal Object R,Legal Object F1,Legal Action P,Legal Action R,Legal Action F1,Legal Basis P,Legal Basis R,Legal Basis F1,Macro-F1,Model
0,www.evenem,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,gold_only_pred
1,open.overh,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,0.867,1.000,0.929,1.000,1.000,1.000,0.986,gold_only_pred
2,www.dnb.nl,1.000,1.000,1.000,1.000,0.909,0.952,1.000,1.000,1.000,1.000,1.000,1.000,0.935,1.000,0.967,0.984,gold_only_pred
3,www.acm.nl,1.000,1.000,1.000,0.986,0.935,0.960,1.000,0.786,0.880,0.704,1.000,0.826,0.972,1.000,0.986,0.930,gold_only_pred
4,kansspelau,1.000,0.727,0.842,1.000,0.731,0.844,0.812,0.812,0.812,0.412,0.875,0.560,1.000,1.000,1.000,0.812,gold_only_pred
5,www.provin,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,0.000,0.000,0.000,0.800,gold_only_pred
6,puc.overhe,1.000,0.892,0.943,0.000,0.000,0.000,0.568,1.000,0.724,1.000,1.000,1.000,0.000,0.000,0.000,0.533,gold_only_pred


,Authority,Recipient P,Recipient R,Recipient F1,Authority P,Authority R,Authority F1,Legal Object P,Legal Object R,Legal Object F1,Legal Action P,Legal Action R,Legal Action F1,Legal Basis P,Legal Basis R,Legal Basis F1,Macro-F1,Model
0,www.evenem,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,silver_only_pred
1,open.overh,0.700,0.778,0.737,1.000,1.000,1.000,1.000,0.733,0.846,0.846,0.846,0.846,1.000,0.306,0.468,0.779,silver_only_pred
2,kansspelau,0.000,0.000,0.000,1.000,0.577,0.732,1.000,0.812,0.897,1.000,1.000,1.000,1.000,0.622,0.767,0.679,silver_only_pred
3,www.provin,1.000,0.857,0.923,1.000,1.000,1.000,0.400,0.800,0.533,0.600,0.900,0.720,0.000,0.000,0.000,0.635,silver_only_pred
4,www.dnb.nl,1.000,0.167,0.286,1.000,1.000,1.000,1.000,0.091,0.167,1.000,0.833,0.909,1.000,0.345,0.513,0.575,silver_only_pred
5,www.acm.nl,0.000,0.000,0.000,0.972,0.896,0.932,1.000,0.750,0.857,1.000,0.947,0.973,1.000,0.049,0.093,0.571,silver_only_pred
6,puc.overhe,0.000,0.000,0.000,0.000,0.000,0.000,1.000,1.000,1.000,1.000,1.000,1.000,0.000,0.000,0.000,0.400,silver_only_pred


,Authority,Recipient P,Recipient R,Recipient F1,Authority P,Authority R,Authority F1,Legal Object P,Legal Object R,Legal Object F1,Legal Action P,Legal Action R,Legal Action F1,Legal Basis P,Legal Basis R,Legal Basis F1,Macro-F1,Model
0,www.dnb.nl,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,silver_to_gold_pred
1,www.evenem,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,silver_to_gold_pred
2,open.overh,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,silver_to_gold_pred
3,www.acm.nl,1.000,1.000,1.000,1.000,0.987,0.993,1.000,0.964,0.982,1.000,1.000,1.000,1.000,0.951,0.975,0.990,silver_to_gold_pred
4,kansspelau,1.000,1.000,1.000,1.000,0.731,0.844,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,0.969,silver_to_gold_pred
5,www.provin,1.000,1.000,1.000,1.000,1.000,1.000,0.429,0.900,0.581,1.000,1.000,1.000,0.000,0.000,0.000,0.716,silver_to_gold_pred
6,puc.overhe,1.000,0.952,0.975,0.000,0.000,0.000,1.000,1.000,1.000,1.000,1.000,1.000,0.000,0.000,0.000,0.595,silver_to_gold_pred
